## QUICK START — Merge Existing Adapter (Skip Training)

**If you already have a trained adapter** on HF Hub (`Semaj90/gemma4-e4b-legal-grpo`) and just need to merge + export GGUF:

1. **Runtime → Change runtime type → G4 GPU** (or A100)
2. Run **cell 3** — Install Unsloth from git (gets PR #4807 ClippableLinear fix)
3. Run **cell 7** — HuggingFace login (needs `HF_TOKEN` in Colab Secrets)
4. **Skip to Section 17** (cell 42+) — Adapter surgery + merge + GGUF export
   - **17a** (cell 43): Pulls cleaned adapter from HF Hub, strips vision/audio if needed
   - **17b** (cell 45): Verifies Unsloth version has ClippableLinear fix
   - **17c** (cell 47): Loads base model + applies text-only LoRA adapter
   - **17d** (cell 49): Quick inference sanity check
   - **17e** (cells 51-53): Merges LoRA → base model (16-bit) → GGUF Q4_K_M export + package
   - **17f** (cell 55): Uploads to HF Hub
5. Download `gemma4-e4b-legal-final-gguf.zip` → deploy locally with Ollama

**Estimated time**: ~15-20 min on G4 (no training, just merge + quantize)

---

**To train from scratch**, run all cells 3-41 sequentially (~1.5-3 hours on G4).

# Gemma 4 E4B Legal AI â€” GRPO Fine-tuning with Unsloth

**Model**: `unsloth/gemma-4-e4b-it-bnb-4bit` (4B active params, multimodal: text + vision + audio)

**Training Method**: GRPO (Group Relative Policy Optimization) â€” RL-based alignment

**Hardware**: Colab **G4 GPU** (RTX Pro 6000 Blackwell, 96GB, 960 BF16 TFLOPs) â€” **recommended**. A100 (80GB) also works. T4 (15GB) feasible with reduced batch.

**Datasets**:
- 60K legal documents (HuggingFace â€” auto-download)
- 200-500 codebase patterns (local upload)
- Legal reward functions for GRPO (citation accuracy, statute references, reasoning chain)

**Target**: RTX 3060 Ti deployment via Ollama GGUF Q4_K_M (~3.5GB VRAM)

**Why Gemma 4 E4B?**:
- Apache 2.0 license (fully open)
- Text + Vision + Audio in ONE model (replaces separate VLM pipeline)
- ~6GB Q4 fits RTX 3060 Ti 8GB with room for embeddings
- Day-0 TRT-LLM support from NVIDIA

**LoRA Adapter Note**: Existing Gemma 3 adapters (5-7hr training) are NOT compatible.
Different architecture = different weight dimensions. Training data (.jsonl) IS reusable.

---

## Prerequisites

1. Extract local datasets:
   ```bash
   cd sveltekit-frontend
   bash ../scripts/dataset-collection/extract-legal-patterns.sh
   ```
2. Verify output: `ls training-datasets/` (7 .jsonl files)
3. Runtime â†’ Change runtime type â†’ **G4 GPU** (recommended) or A100

**No PTX issues** â€” Unsloth uses Triton kernels that JIT-compile at runtime for any GPU architecture. No precompiled PTX/SASS binaries needed.

**Training time**: ~1.5-3 hours on G4, ~3-5 hours on A100
**Output size**: ~3 GB (Q4_K_M GGUF)
**Cost**: ~$6-10 (Colab Pro+ G4)

## 1. Setup

In [ ]:
# Install Unsloth and essential dependencies (Excluding mergekit to avoid Pydantic errors)
!pip uninstall unsloth mergekit mergekit-moe -y 2>/dev/null || true
%pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install bitsandbytes accelerate peft trl transformers "datasets>=4.2.0,<4.4.0" huggingface_hub pillow llm_blender weave

In [ ]:
import torch
import json
import re
from pathlib import Path
from datasets import load_dataset, concatenate_datasets, Dataset

# Gemma 4 E4B is MULTIMODAL (text + vision + audio).
# FastLanguageModel mishandles multimodal saves â€” adapter loses language_model tensors.
# FastVisionModel properly handles all sub-models.
try:
    from unsloth import FastVisionModel, is_bfloat16_supported
    MODEL_LOADER = FastVisionModel
    print("Using FastVisionModel (multimodal-aware loader)")
except ImportError:
    from unsloth import FastLanguageModel, is_bfloat16_supported
    MODEL_LOADER = FastLanguageModel
    print("WARNING: FastVisionModel not available, falling back to FastLanguageModel")
    print("  This may cause adapter save to MISS language_model tensors!")

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

GPU_PROFILE = {
    "name": None,
    "vram_gb": 0,
    "bf16_tflops": "unknown",
    "recommended_batch_size": 1,
    "recommended_num_generations": 2,
    "recommended_max_completion": 128,
    "recommended_max_seq_length": 2048,
    "recommended_grad_accum": 8,
}

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    cc = torch.cuda.get_device_capability(0)

    GPU_PROFILE["name"] = gpu_name
    GPU_PROFILE["vram_gb"] = round(vram_gb, 1)

    print(f"GPU: {gpu_name}")
    print(f"VRAM: {vram_gb:.1f} GB")
    print(f"Compute Capability: {cc[0]}.{cc[1]}")

    if cc[0] >= 10:
        print("  Blackwell architecture confirmed (sm_100+)")
    elif cc[0] >= 9:
        print("  Hopper architecture (sm_90+)")
    elif cc[0] >= 8:
        print("  Ampere architecture (sm_80+)")

    if vram_gb >= 90:
        GPU_PROFILE.update({
            "bf16_tflops": "~960",
            "recommended_batch_size": 4,
            "recommended_num_generations": 2,
            "recommended_max_completion": 128,
            "recommended_max_seq_length": 2048,
            "recommended_grad_accum": 2,
        })
        print("\nG4 / Blackwell GPU detected â€” use speed-first GRPO defaults.")
    elif vram_gb >= 70:
        GPU_PROFILE.update({
            "bf16_tflops": "~624",
            "recommended_batch_size": 2,
            "recommended_num_generations": 2,
            "recommended_max_completion": 128,
            "recommended_max_seq_length": 2048,
            "recommended_grad_accum": 4,
        })
        print("\nA100-80GB detected â€” full GRPO available.")
    elif vram_gb >= 35:
        GPU_PROFILE.update({
            "bf16_tflops": "~312",
            "recommended_batch_size": 1,
            "recommended_num_generations": 2,
            "recommended_max_completion": 128,
            "recommended_max_seq_length": 2048,
            "recommended_grad_accum": 8,
        })
        print("\nA100-40GB detected â€” standard GRPO config.")
    else:
        GPU_PROFILE.update({
            "recommended_batch_size": 1,
            "recommended_num_generations": 1,
            "recommended_max_completion": 96,
            "recommended_max_seq_length": 1024,
            "recommended_grad_accum": 8,
        })
        print(f"\nWARNING: {vram_gb:.1f}GB may be tight for GRPO.")

    print("\nRecommended training profile:")
    print(json.dumps(GPU_PROFILE, indent=2))
else:
    print("No CUDA GPU detected.")

## 2. Model Configuration

In [ ]:
import torch

# Gemma 4 E4B â€” Fast Pilot Profile
MODEL_NAME = "unsloth/gemma-4-E4B-it-unsloth-bnb-4bit"
MAX_SEQ_LENGTH = 4096

# LoRA configuration
LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0

# PHASE 1: FAST PILOT CONFIGURATION
# Phase 1: 2 gen, 256 completion, 1K prompts â†’ ~1-2 hours â†’ validate rewards
# Phase 2: Scale up to 4 gen, 512 completion, full dataset after rewards proven
NUM_GENERATIONS = 2
BATCH_SIZE = 4
GRAD_ACCUM = 2
MAX_COMPLETION = 256

print(f"Model: {MODEL_NAME}")
print(f"Phase 1 Pilot: {NUM_GENERATIONS} generations")
print(f"Effective batch: {BATCH_SIZE * GRAD_ACCUM}")
print(f"Max completion tokens: {MAX_COMPLETION}")

In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Login to HuggingFace â€” required for gated model access
# Set your token in Colab: Settings > Secrets > HF_TOKEN
try:
    hf_token = userdata.get('HF_TOKEN')
    login(hf_token)
    print("Logged in via Colab secret HF_TOKEN")
except Exception:
    # Fallback: manual token entry
    print("HF_TOKEN secret not found. Enter your token manually:")
    login()
print("HuggingFace login successful!")

## 3. Load Model + Add LoRA

In [ ]:
# Cell intentionally left minimal — the ClippableLinear patch has been moved
# into the model loading cell below. Unsloth 2026.4.2 already has "Fast Gemma4
# patching" — we only need to provide the missing import, not replace the class.
print("ClippableLinear fix: integrated into model loading cell (next cell)")

In [ ]:
import sys
import importlib.util
from pathlib import Path

print(f"Loading {MODEL_NAME}...\n")

# === FIX: Inject Gemma4ClippableLinear into Unsloth's compiled module ===
# Unsloth 2026.4.2 compiles a custom gemma4 module that references bare
# `Gemma4ClippableLinear` without importing it — causes NameError on load.
#
# Strategy: Pre-load the compiled module, inject the class from transformers,
# then let from_pretrained find it already in sys.modules.

from transformers.models.gemma4 import modeling_gemma4
compiled_path = Path("/content/unsloth_compiled_cache/unsloth_compiled_module_gemma4.py")

if compiled_path.exists() and hasattr(modeling_gemma4, 'Gemma4ClippableLinear'):
    compiled_mod_name = "unsloth_compiled_module_gemma4"

    # Load the compiled module into sys.modules
    spec = importlib.util.spec_from_file_location(compiled_mod_name, str(compiled_path))
    mod = importlib.util.module_from_spec(spec)

    # Inject the class BEFORE exec_module runs the module code
    mod.Gemma4ClippableLinear = modeling_gemma4.Gemma4ClippableLinear
    mod.nn = __import__('torch').nn
    mod.torch = __import__('torch')

    sys.modules[compiled_mod_name] = mod
    try:
        spec.loader.exec_module(mod)
        print(f"Injected Gemma4ClippableLinear into {compiled_mod_name}")
    except Exception as e:
        print(f"Compiled module pre-load failed: {e}")
        print("Falling back to builtins injection...")
        import builtins
        builtins.Gemma4ClippableLinear = modeling_gemma4.Gemma4ClippableLinear
else:
    # Compiled cache doesn't exist yet or no ClippableLinear — inject via builtins
    print("No compiled cache found — injecting via builtins fallback")
    import builtins
    if hasattr(modeling_gemma4, 'Gemma4ClippableLinear'):
        builtins.Gemma4ClippableLinear = modeling_gemma4.Gemma4ClippableLinear
    else:
        builtins.Gemma4ClippableLinear = _PATCHED_CLIPPABLE

# Now load the model
model, tokenizer = MODEL_LOADER.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,  # Auto-detect (bfloat16 if supported)
)

print(f"\nLoaded: {MODEL_NAME}")
print(f"Loader: {MODEL_LOADER.__name__}")
print(f"BFloat16: {is_bfloat16_supported()}")

# Verify model architecture
arch = model.config.architectures[0] if hasattr(model.config, 'architectures') else "unknown"
print(f"Architecture: {arch}")
if "Conditional" in arch or "Vision" in arch:
    print("  Multimodal model detected — FastVisionModel recommended")

In [ ]:
print("Adding LoRA adapters...\n")

# Use Unsloth's native FastVisionModel parameters to control which sub-models get LoRA.
# finetune_vision_layers=False excludes vision_tower (and audio_tower inherits this).
# This avoids PEFT exclude_modules issues and Unsloth post_patch_model traversal errors.

model = MODEL_LOADER.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=True,
)

model.print_trainable_parameters()

# === VERIFY ===
lora_modules = [name for name, p in model.named_parameters() if "lora_" in name and p.requires_grad]
lang_count = sum(1 for n in lora_modules if "language_model" in n)
vision_count = sum(1 for n in lora_modules if "vision_tower" in n)
audio_count = sum(1 for n in lora_modules if "audio_tower" in n)

print(f"\nLoRA target verification:")
print(f"  language_model: {lang_count} trainable tensors {'OK' if lang_count > 0 else 'MISSING!'}")
print(f"  vision_tower:   {vision_count} trainable tensors {'(excluded)' if vision_count == 0 else 'WARNING'}")
print(f"  audio_tower:    {audio_count} trainable tensors {'(excluded)' if audio_count == 0 else 'WARNING'}")

if lang_count == 0:
    raise RuntimeError("FATAL: No language_model LoRA tensors! Check get_peft_model config.")

## 4. Load Legal Datasets

In [ ]:
def standardize_text(example):
    if 'text' not in example:
        return example
    if isinstance(example['text'], list):
        example['text'] = ' '.join([
            item['value'] if isinstance(item, dict) and 'value' in item else str(item)
            for item in example['text']
        ])
    elif not isinstance(example['text'], str):
        example['text'] = str(example['text'])
    return example

print("Loading HuggingFace legal datasets...\n")

# 1. FineTome
print("[1/6] FineTome...")
dataset1 = load_dataset("mlabonne/FineTome-100k", split="train[:10000]")
dataset1 = dataset1.rename_column('conversations', 'text')
print(f"  {len(dataset1):,}")

# 2. GSM8K
print("[2/6] GSM8K...")
dataset2 = load_dataset("openai/gsm8k", "main", split="train[:5000]")
dataset2 = dataset2.rename_column('question', 'text')
print(f"  {len(dataset2):,}")

# 3. Pile of Law
print("[3/6] Pile of Law...")
pile_of_law = load_dataset("lamblamb/pile_of_law_subset", split="train[:20000]")
print(f"  {len(pile_of_law):,}")

# 4. LEDGAR (contract provisions â€” lex_glue collection)
print("[4/6] LEDGAR...")
try:
    ledgar = load_dataset("lex_glue", "ledgar", split="train[:10000]")
    print(f"  {len(ledgar):,}")
except Exception as e:
    print(f"  SKIPPED (script-based loading unsupported in datasets 4.x): {e}")
    ledgar = None

# 5. Case Hold (legal reasoning â€” lighteval/lexglue parquet version)
print("[5/6] Case Hold...")
try:
    case_hold = load_dataset("lighteval/lexglue", name="case_hold", split="train[:5000]")
    if 'input' in case_hold.column_names:
        case_hold = case_hold.rename_column('input', 'text')
    print(f"  {len(case_hold):,}")
except Exception as e:
    print(f"  SKIPPED: {e}")
    case_hold = None

# 6. SCOTUS (Supreme Court opinions â€” lighteval/lexglue parquet version)
print("[6/6] SCOTUS...")
try:
    scotus = load_dataset("lighteval/lexglue", name="scotus", split="train[:5000]")
    if 'input' in scotus.column_names:
        scotus = scotus.rename_column('input', 'text')
    print(f"  {len(scotus):,}")
except Exception as e:
    print(f"  SKIPPED: {e}")
    scotus = None

# Standardize all datasets to have a 'text' column
print("\nStandardizing...")
legal_datasets = []
for ds in [dataset1, dataset2, pile_of_law, ledgar, case_hold, scotus]:
    if ds is None:
        continue
    if 'text' in ds.column_names:
        ds = ds.select_columns(['text']).map(standardize_text, num_proc=4)
    elif 'input' in ds.column_names:
        ds = ds.rename_column('input', 'text').select_columns(['text']).map(standardize_text, num_proc=4)
    else:
        cols = ds.column_names
        for col in cols:
            if isinstance(ds[0][col], str) and len(ds[0][col]) > 20:
                ds = ds.rename_column(col, 'text').select_columns(['text']).map(standardize_text, num_proc=4)
                break
        else:
            print(f"  Warning: skipping dataset with columns {cols}")
            continue
    legal_datasets.append(ds)

legal_dataset = concatenate_datasets(legal_datasets)
print(f"\nLegal datasets: {len(legal_dataset):,} examples")

## 5. Upload Codebase Datasets

**Before running**: Extract training data locally:
```bash
cd sveltekit-frontend
bash ../scripts/dataset-collection/extract-legal-patterns.sh
```

Upload all 7 .jsonl files from `training-datasets/`

In [ ]:
from google.colab import files

print("Upload your training-datasets/*.jsonl files")
print("(Select all 7 files at once)\n")

uploaded = files.upload()

codebase_patterns = []
for filename, content in uploaded.items():
    if filename.endswith('.jsonl'):
        print(f"Loading {filename}...")
        lines = content.decode('utf-8').strip().split('\n')
        for line in lines:
            if line.strip():
                try:
                    codebase_patterns.append(json.loads(line))
                except json.JSONDecodeError:
                    continue

print(f"\nCodebase patterns: {len(codebase_patterns):,} examples")

## 6. Prepare GRPO Prompt Dataset

GRPO needs a dataset of **prompts** (not prompt-response pairs).
The model generates multiple completions per prompt, then a reward function scores them.

In [ ]:
grpo_prompts = []

# Category 1: Legal analysis prompts (from legal dataset text snippets)
for i, example in enumerate(legal_dataset):
    text = example.get('text', '')
    if len(text) < 50:
        continue

    snippet = text[:200].strip()

    if any(kw in text.lower() for kw in ['statute', 'u.s.c', 'code', 'section']):
        prompt = f"Analyze the following legal statute and explain its implications:\n\n{snippet}..."
    elif any(kw in text.lower() for kw in ['court', 'judge', 'ruling', 'opinion']):
        prompt = f"Summarize the key holdings and reasoning in this court opinion:\n\n{snippet}..."
    elif any(kw in text.lower() for kw in ['contract', 'agreement', 'party', 'clause']):
        prompt = f"Review this contract provision and identify key legal terms and obligations:\n\n{snippet}..."
    elif any(kw in text.lower() for kw in ['evidence', 'testimony', 'witness']):
        prompt = f"Analyze this evidence description for a legal investigation:\n\n{snippet}..."
    else:
        prompt = f"Provide legal analysis of the following:\n\n{snippet}..."

    grpo_prompts.append({
        "prompt": prompt,
        "reference_text": text[:500],
    })

    if len(grpo_prompts) >= 3000:
        break

# Category 2: Codebase pattern prompts
for pattern in codebase_patterns:
    text = pattern.get('text', '')
    if len(text) < 30:
        continue
    prompt = f"Explain the following technical concept:\n\n{text[:150]}..."
    grpo_prompts.append({"prompt": prompt, "reference_text": text[:500]})

# Category 3: Freeform legal reasoning prompts (no reference â€” tests generalization)
freeform_prompts = [
    "What are the elements of a negligence claim under common law?",
    "Explain the difference between civil and criminal burden of proof.",
    "Summarize the key provisions of 42 U.S.C. Section 1983.",
    "What is the chain of custody requirement for physical evidence?",
    "Describe the hearsay rule and its major exceptions under the Federal Rules of Evidence.",
    "What factors do courts consider when deciding motions for summary judgment?",
    "Explain how Daubert v. Merrell Dow applies to expert witness testimony.",
    "What are the Miranda rights and when must they be given?",
    "Describe the differences between express and implied contracts.",
    "What constitutes a valid search warrant under the Fourth Amendment?",
    "Explain the doctrine of res judicata and collateral estoppel.",
    "What are the remedies available in a breach of contract action?",
    "Describe the standard for granting preliminary injunctive relief.",
    "What is the attorney-client privilege and when can it be waived?",
    "Explain the concept of joint and several liability in tort law.",
]

for p in freeform_prompts:
    grpo_prompts.append({"prompt": p, "reference_text": ""})

# Category 4: Tool calling + web search prompts (trains agentic behavior)
tool_calling_prompts = [
    'I need the legal definition of "habeas corpus". Use the glossary_search tool to look it up.',
    "Define the term 'stare decisis' using the glossary_search tool.",
    "Search our evidence database for documents related to the Fourth Amendment violation claims.",
    "Use rag_search to find evidence about chain of custody failures in this case.",
    "Search the web for the latest Supreme Court rulings on qualified immunity from 2024-2025.",
    "Use web_search to find recent case law on digital evidence admissibility.",
    "Show me the knowledge graph connections for this case â€” what evidence links to what?",
    "Drill down into 42 U.S.C. Â§ 1983 â€” find the full text and related authorities.",
    "Find similar cases to this one involving excessive force by law enforcement.",
    "First search our documents for qualified immunity precedents, then search the web for any recent updates.",
]

for p in tool_calling_prompts:
    grpo_prompts.append({"prompt": p, "reference_text": ""})

grpo_dataset = Dataset.from_list(grpo_prompts)

# PHASE 1: Select 1,000 prompts for fast pilot validation
# Phase 2: Remove this line to use full dataset after rewards are proven
grpo_dataset = grpo_dataset.select(range(min(1000, len(grpo_dataset))))

print(f"Fast Pilot GRPO dataset: {len(grpo_dataset):,} prompts")
print(f"  (Full pool: {len(grpo_prompts):,} â€” trimmed to 1,000 for Phase 1)")
print(f"Example: {grpo_dataset[0]['prompt'][:100]}...")

## 7. Define Legal Reward Functions

GRPO uses reward functions to score model completions.

**Phase 1 (Fast Pilot)**: Single combined function with 5 high-speed signals â€” optimized for training throughput (it/s).
- Citation accuracy (0.25) â€” valid Bluebook/USC/CFR formats
- Reasoning logic (0.25) â€” logical connectors (therefore, because, pursuant to)
- Legal formatting (0.20) â€” numbered lists, paragraph structure
- Hallucination guard (0.15) â€” penalize fabricated case names
- Length efficiency (0.15) â€” target 100-250 words

**Phase 2 (Production)**: Expand to 8 separate functions with RAG/KAG/DAG context, web search grounding, and tool calling format signals. See the full 8-function version in the commented block below.

In [ ]:
import re
import numpy as np

# High-speed regex patterns for reward signals
CITATION_RE = re.compile(r'\d+\s+[A-Z][a-z]*\.?\s*(?:2d|3d|4th|Supp\.?)?\s*\d+', re.IGNORECASE)
STATUTE_RE = re.compile(r'\d+\s+U\.?S\.?C\.?\s*(?:Â§|Section)?\s*\d+', re.IGNORECASE)
REASONING_WORDS = ['because', 'therefore', 'thus', 'consequently', 'accordingly', 'pursuant to', 'under', 'based on']

def combined_legal_reward(completions, **kwargs):
    """Fast-path Reward Function: Optimized for training speed (it/s).

    5 signals weighted to sum to 1.0:
    - Citation (0.25): Bluebook, U.S.C., CFR patterns
    - Reasoning (0.25): Logical connectors
    - Formatting (0.20): Numbered lists, paragraph structure
    - Anti-hallucination (0.15): Penalize fabricated case names
    - Length efficiency (0.15): Target 100-250 words
    """
    rewards = []
    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else str(completion)
        text_lower = text.lower()

        # 1. Citation Signal (0.25)
        s_citation = min(1.0, len(CITATION_RE.findall(text)) * 0.5 + len(STATUTE_RE.findall(text)) * 0.5)

        # 2. Reasoning Logic (0.25)
        s_reasoning = min(1.0, sum(1 for w in REASONING_WORDS if w in text_lower) * 0.25)

        # 3. Legal Formatting (0.20)
        s_format = 0.5 if re.search(r'\d+\.\s', text) else 0.0
        if text.count('\n') >= 2: s_format += 0.5

        # 4. Hallucination Guard (0.15)
        s_halluc = 1.0
        if re.search(r'(?:Smith|Jones|Doe)\s+v\.\s+(?:Smith|Jones|Doe)', text): s_halluc -= 0.5

        # 5. Length Efficiency (0.15)
        # Target: 100-250 words for concise legal accuracy
        words = len(text.split())
        if 100 <= words <= 250: s_len = 1.0
        elif words < 100: s_len = words / 100
        else: s_len = 0.5

        total = (0.25*s_citation + 0.25*s_reasoning + 0.20*s_format + 0.15*s_halluc + 0.15*s_len)
        rewards.append(total)
    return rewards

# === Verify reward function works ===
test_completion = [[{"content": "Under 42 U.S.C. Section 1983, the plaintiff must demonstrate deprivation of constitutional rights. Therefore, the court must establish jurisdiction pursuant to 28 U.S.C. Section 1331.\n\n1. Federal question jurisdiction exists.\n2. Qualified immunity analysis follows the Saucier two-step test.\n3. The statute of limitations varies by state."}]]
print(f"Fast-path Legal Reward test: {combined_legal_reward(test_completion)[0]:.3f}")
print("Fast-path Legal Reward Function initialized (5 high-speed signals).")

# ============================================================================
# PHASE 2: Full 8-function reward suite (uncomment for production training)
# ============================================================================
# RAG_PATTERNS = [
#     r'(?:retrieved|found in|according to|from the knowledge base|from our database)',
#     r'(?:document|evidence|exhibit)\s+(?:ID|#|number)?\s*[\w-]+',
#     r'(?:relevance|confidence|similarity)\s*(?:score|rating)?\s*:?\s*\d',
# ]
# KAG_PATTERNS = [
#     r'(?:knowledge graph|graph database|neo4j|relationship)',
#     r'(?:entity|entities)\s+(?:extracted|identified|found|linked)',
# ]
# DAG_PATTERNS = [
#     r'(?:dependency|dependencies|prerequisite|depends on|required by)',
#     r'(?:authority chain|citation chain|precedent chain)',
# ]
#
# def reward_citation_accuracy(completions, **kwargs): ...
# def reward_reasoning_chain(completions, **kwargs): ...
# def reward_rag_kag_dag_context(completions, **kwargs): ...
# def reward_legal_formatting(completions, **kwargs): ...
# def reward_web_search_grounding(completions, **kwargs): ...
# def reward_tool_calling_format(completions, **kwargs): ...
# def reward_anti_hallucination(completions, **kwargs): ...
# def reward_length_quality(completions, **kwargs): ...
#
# def combined_legal_reward_full(completions, **kwargs):
#     """Phase 2: 8-signal weighted combination (slower, higher quality)"""
#     r1 = reward_citation_accuracy(completions, **kwargs)
#     r2 = reward_reasoning_chain(completions, **kwargs)
#     r3 = reward_rag_kag_dag_context(completions, **kwargs)
#     r4 = reward_legal_formatting(completions, **kwargs)
#     r5 = reward_web_search_grounding(completions, **kwargs)
#     r6 = reward_tool_calling_format(completions, **kwargs)
#     r7 = reward_anti_hallucination(completions, **kwargs)
#     r8 = reward_length_quality(completions, **kwargs)
#     return [
#         0.15*a + 0.15*b + 0.15*c + 0.12*d + 0.12*e + 0.10*f + 0.10*g + 0.11*h
#         for a, b, c, d, e, f, g, h in zip(r1, r2, r3, r4, r5, r6, r7, r8)
#     ]

## 8. Configure GRPO Training

Mock phantom dependencies that TRL tries to import at init time (weave, mergekit, llm_blender).
Then configure GRPOConfig with speed-first settings.

In [ ]:
import sys
import os
import importlib.util
from unittest.mock import MagicMock

def mock_module(name):
    """Deep mock a module to satisfy complex library imports."""
    if name not in sys.modules:
        module = MagicMock()
        spec = importlib.util.spec_from_loader(name, loader=None)
        module.__spec__ = spec
        sys.modules[name] = module
    return sys.modules[name]

# === Fix TRL phantom dependency crashes in Colab ===
# Colab has broken/partial packages (weave, llm_blender, mergekit) that TRL detects
# as present but can't actually import. mock_module stubs them in sys.modules.
mock_module("mergekit")
mock_module("mergekit.config")
mock_module("mergekit.merge")
mock_module("weave")
mock_module("weave.trace")
mock_module("weave.trace.context")

# Fix TRL internal availability checks — disable optional deps
import trl.import_utils as _trl_iu
_keep = {'is_peft_available', 'is_accelerate_available', 'is_torch_available', 'is_datasets_available'}
for _attr in dir(_trl_iu):
    if _attr.startswith('is_') and _attr.endswith('_available') and _attr not in _keep:
        if callable(getattr(_trl_iu, _attr, None)):
            setattr(_trl_iu, _attr, lambda: False)

# === Now TRL imports cleanly ===
from trl import GRPOConfig, GRPOTrainer
import torch
from unsloth import is_bfloat16_supported

print(f"TRL {trl.__version__} imported successfully")

# Speed-first config:
# effective batch = BATCH_SIZE * GRAD_ACCUM
# num_generations must divide effective batch
EFFECTIVE_BATCH = BATCH_SIZE * GRAD_ACCUM

if EFFECTIVE_BATCH % 2 == 0:
    CURRENT_NUM_GEN = 2
else:
    CURRENT_NUM_GEN = 1

grpo_config = GRPOConfig(
    output_dir="./gemma4-e4b-legal-grpo-output",

    # GRPO-specific (auto-scaled)
    num_generations=CURRENT_NUM_GEN,
    max_completion_length=MAX_COMPLETION,
    max_prompt_length=256,

    # Training hyperparams — 1 epoch for pilot
    num_train_epochs=1,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=5e-6,

    # Precision
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),

    # Checkpointing
    logging_steps=10,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    warmup_steps=30,

    # Optimizer
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    max_grad_norm=0.5,

    # Speed optimizations — lighter dataloader for Colab stability
    dataloader_num_workers=2,
    dataloader_pin_memory=True,

    # FIX: Gemma 4 Day-Zero Bug — mm_token_type_ids required even for text-only
    # Gemma 4's processor adds mm_token_type_ids to tokenized inputs.
    # If remove_unused_columns=True (default), the trainer drops this column,
    # causing "mm_token_type_ids is required" errors during forward pass.
    remove_unused_columns=False,

    # Misc
    seed=42,
    report_to="none",
)

print(f"GRPO Training Config initialized.")
print(f"  Generations: {CURRENT_NUM_GEN}")
print(f"  Effective Batch: {EFFECTIVE_BATCH}")
print(f"  Max prompt: {grpo_config.max_prompt_length}")
print(f"  Max completion: {grpo_config.max_completion_length}")
print(f"  Precision: {'bfloat16' if is_bfloat16_supported() else 'float16'}")
print(f"  remove_unused_columns: False (Gemma 4 mm_token_type_ids fix)")

## 9. Initialize GRPO Trainer

In [ ]:
import inspect
from trl import GRPOTrainer

# Fix: TRL sets model.warnings_issued["estimate_tokens"] but PEFT/Unsloth models
# don't have this attribute. Add it before GRPOTrainer.__init__ accesses it.
if not hasattr(model, 'warnings_issued'):
    model.warnings_issued = {}

# Keep trainer args aligned with the actual run config
EFFECTIVE_BATCH = BATCH_SIZE * GRAD_ACCUM
assert EFFECTIVE_BATCH % CURRENT_NUM_GEN == 0, (
    f"num_generations ({CURRENT_NUM_GEN}) must divide effective batch ({EFFECTIVE_BATCH})"
)

print(f"Trainer setup: gen={CURRENT_NUM_GEN}, batch={BATCH_SIZE}, "
      f"grad_accum={GRAD_ACCUM}, eff_batch={EFFECTIVE_BATCH}, "
      f"max_completion={MAX_COMPLETION}")

# Auto-detect GRPOTrainer API (handles config= vs args=, tokenizer= vs processing_class=)
trainer_kwargs = {
    "model": model,
    "reward_funcs": [combined_legal_reward],
    "train_dataset": grpo_dataset,
}

sig = inspect.signature(GRPOTrainer.__init__)
_params = set(sig.parameters.keys())

if 'config' in _params:
    trainer_kwargs['config'] = grpo_config
elif 'args' in _params:
    trainer_kwargs['args'] = grpo_config

if 'processing_class' in _params:
    trainer_kwargs['processing_class'] = tokenizer
elif 'tokenizer' in _params:
    trainer_kwargs['tokenizer'] = tokenizer

trainer = GRPOTrainer(**trainer_kwargs)

print(f"GRPO Trainer initialized (TRL {trl.__version__})")
print(f"  Reward: combined_legal_reward (5 fast-path signals)")
print(f"  Generations: {trainer.args.num_generations}")
print(f"  Training prompts: {len(grpo_dataset):,}")

## 10. Train (3-5 hours)

In [ ]:
print("=" * 70)
print("GRPO TRAINING START")
print("=" * 70)
print(f"Model: Gemma 4 E4B")
print(f"Method: GRPO ({trainer.args.num_generations} generations/prompt)")
print(f"Prompts: {len(grpo_dataset):,}")
print(f"Epochs: {grpo_config.num_train_epochs}")
print(f"Estimated time: 1.5-3 hours (G4 Blackwell Edition)\n")

trainer_stats = trainer.train()

print("\n" + "=" * 70)
print("GRPO TRAINING COMPLETE")
print("=" * 70)
runtime = trainer_stats.metrics['train_runtime']
print(f"Time: {runtime:.0f}s ({runtime/3600:.1f} hours)")
print(f"Samples/sec: {trainer_stats.metrics['train_samples_per_second']:.2f}")

## 11. Test Inference

In [ ]:
from transformers import TextStreamer

MODEL_LOADER.for_inference(model)

test_prompts = [
    "Explain the elements of a negligence claim and cite relevant case law.",
    "What is the chain of custody requirement for digital evidence?",
    "Describe the RAG evidence upload pipeline in a legal AI system.",
    "Analyze 42 U.S.C. Section 1983 and its application to police misconduct cases.",
]

text_streamer = TextStreamer(tokenizer, skip_prompt=True)

for prompt in test_prompts:
    print("\n" + "=" * 70)
    print(f"Prompt: {prompt}")
    print("=" * 70)

    # Gemma 4 E4B is multimodal â€” content must be a list of typed blocks,
    # not a plain string. The processor iterates content expecting dicts
    # with "type" keys; plain strings cause: string indices must be integers
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    model.generate(
        input_ids=inputs,
        streamer=text_streamer,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
    )
    print()

## 12. Save LoRA Adapters

In [ ]:
# === Output 1: Raw LoRA adapters (always works, smallest) ===
model.save_pretrained("gemma4-e4b-legal-grpo-lora")
tokenizer.save_pretrained("gemma4-e4b-legal-grpo-lora")
print("1/3  LoRA adapters saved: gemma4-e4b-legal-grpo-lora/")

# === VERIFY: Ensure language_model tensors were actually saved ===
from safetensors.torch import load_file as _load_st
_saved = _load_st("gemma4-e4b-legal-grpo-lora/adapter_model.safetensors")
_lang = sum(1 for n in _saved if "language_model" in n)
_vis = sum(1 for n in _saved if "vision_tower" in n)
_aud = sum(1 for n in _saved if "audio_tower" in n)
print(f"  Saved tensors: {len(_saved)} total")
print(f"    language_model: {_lang} {'OK' if _lang > 0 else 'MISSING â€” ADAPTER IS BROKEN!'}")
print(f"    vision_tower:   {_vis} {'(excluded)' if _vis == 0 else '(included)'}")
print(f"    audio_tower:    {_aud} {'(excluded)' if _aud == 0 else '(included)'}")
if _lang == 0:
    print("\n  *** CRITICAL: language_model LoRA weights NOT saved! ***")
    print("  The adapter is useless for text generation.")
    print("  Re-run cell 8 with exclude_modules or regex target_modules.")
del _saved, _lang, _vis, _aud

# === Output 2: Merged BF16 for TRT-LLM engine building (~8GB) ===
try:
    model.save_pretrained_merged(
        "gemma4-e4b-legal-merged-bf16",
        tokenizer,
        save_method="merged_bf16",
    )
    print("\n2/3  Merged BF16 saved: gemma4-e4b-legal-merged-bf16/ (~8GB)")
    print("     TRT-LLM: trtllm-build --checkpoint_dir gemma4-e4b-legal-merged-bf16/")
except Exception as e:
    print(f"\n2/3  BF16 merge failed: {e}")
    print("     Use LoRA adapters directly with TRT-LLM:")
    print("     trtllm-build --checkpoint_dir <base_model> --lora_dir gemma4-e4b-legal-grpo-lora/")

print("\n3/3  GGUF export is in the next cell (cell 28)")
print()
print("=== Export Summary ===")
print("  Ollama:   cell 28 â†’ GGUF Q4_K_M (~3GB) â€” handles merge internally")
print("  TRT-LLM:  gemma4-e4b-legal-merged-bf16/ â†’ trtllm-build")
print("  Raw LoRA:  gemma4-e4b-legal-grpo-lora/ â†’ HF Hub / manual merge")

In [ ]:
# Zip LoRA adapters for download + Google Drive backup
!zip -r gemma4-legal-adapters.zip gemma4-e4b-legal-grpo-lora/
print("Adapters zipped: gemma4-legal-adapters.zip")

# Trigger browser download
from google.colab import files
try:
    files.download('gemma4-legal-adapters.zip')
    print("Triggering browser download...")
except Exception as e:
    print(f"Manual download trigger failed (normal in some environments): {e}")

# Backup to Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    !cp gemma4-legal-adapters.zip /content/drive/MyDrive/
    print("Adapters backed up to Google Drive (MyDrive root).")
except Exception as e:
    print(f"Google Drive backup skipped: {e}")

## 13. Export to GGUF for Ollama

Direct GGUF export via Unsloth â€” creates Ollama-ready model file.
Falls back to forced_merged_4bit â†’ GGUF if direct export fails (common with multimodal models).

In [ ]:
import os

# === Output 3: GGUF Q4_K_M for Ollama (~3GB) ===
# IMPORTANT: Ollama's ADAPTER directive does NOT support Gemma 4 SafeTensors.
# Must use merged GGUF approach: FROM ./model.gguf (no ADAPTER line).
#
# FIX: save_pretrained_gguf needs config.json in the output dir.
# For multimodal models, this file isn't auto-generated.
# Solution: save_pretrained_merged("merged_16bit") FIRST to create config.json,
# then save_pretrained_gguf() can find it.

os.makedirs("gemma4-e4b-legal", exist_ok=True)
merged_16bit_path = "gemma4-e4b-legal-merged-16bit"

print("Step 1: Creating merged 16-bit model (generates config.json)...")
try:
    model.save_pretrained_merged(
        merged_16bit_path,
        tokenizer,
        save_method="merged_16bit",
    )
    print(f"  Merged 16-bit saved to {merged_16bit_path}/")

    # Verify config.json exists
    config_path = os.path.join(merged_16bit_path, "config.json")
    if os.path.exists(config_path):
        print(f"  config.json exists ({os.path.getsize(config_path)} bytes)")
    else:
        print("  WARNING: config.json not found in merged output!")
except Exception as e:
    print(f"  16-bit merge failed: {e}")
    merged_16bit_path = None

print("\nStep 2: Exporting to GGUF Q4_K_M (Ollama-ready)...")
print("This may take 5-15 minutes (quantize)...\n")

try:
    model.save_pretrained_gguf(
        "gemma4-e4b-legal",
        tokenizer,
        quantization_method="q4_k_m",
    )
    print("\nGGUF saved: gemma4-e4b-legal/")
    gguf_files = [f for f in os.listdir("gemma4-e4b-legal") if f.endswith('.gguf')]
    for gf in gguf_files:
        size_gb = os.path.getsize(f"gemma4-e4b-legal/{gf}") / 1024**3
        print(f"  {gf}: {size_gb:.1f} GB")
except Exception as e:
    print(f"GGUF Export failed: {e}")
    print("\nTrying fallback: forced_merged_4bit → GGUF...")

    try:
        merged_path = "gemma4-e4b-legal-merged-4bit"
        os.makedirs(merged_path, exist_ok=True)
        model.save_pretrained_merged(merged_path, tokenizer, save_method="forced_merged_4bit")
        print(f"  4-bit merge saved to {merged_path}")

        model.save_pretrained_gguf("gemma4-e4b-legal-gguf-export", tokenizer, quantization_method="q4_k_m")
        print("  GGUF from merged model saved!")
    except Exception as e2:
        print(f"  Fallback also failed: {e2}")
        print("\n  Try manual llama.cpp conversion (next cell)")
        print("  Or use merged_16bit/ directly with llama.cpp convert_hf_to_gguf.py")
        if merged_16bit_path:
            print(f"  Merged 16-bit model available at: {merged_16bit_path}/")

print("\n=== DEPLOYMENT NOTE ===")
print("Ollama ADAPTER directive does NOT support Gemma 4.")
print("Use merged GGUF: ollama create gemma4-legal -f Modelfile")
print("Where Modelfile has: FROM ./unsloth.Q4_K_M.gguf (no ADAPTER line)")

In [ ]:
import os

# === Manual llama.cpp GGUF Conversion (fallback if Unsloth export fails) ===
# NOTE: llama.cpp's convert_hf_to_gguf.py may not support Gemma4Model yet.
# This cell tries it anyway — if it fails, the Unsloth GGUF from the previous cell
# is the only viable path. Check https://github.com/ggerganov/llama.cpp/issues for updates.

# Use the merged 16-bit model (has all weights + config.json)
# LoRA-only adapters won't work with convert_hf_to_gguf
merged_path = "gemma4-e4b-legal-merged-16bit"

if not os.path.exists(merged_path):
    print(f"ERROR: {merged_path}/ not found. Run the previous cell first.")
    print("The merged 16-bit model is required for llama.cpp conversion.")
else:
    # Setup llama.cpp if not present
    if not os.path.exists("llama.cpp"):
        print("Cloning llama.cpp for manual conversion...")
        !git clone --depth 1 https://github.com/ggerganov/llama.cpp.git
        %pip install -r llama.cpp/requirements.txt

    # Attempt manual conversion from merged model
    print(f"\nConverting {merged_path}/ to GGUF using llama.cpp...")
    !python llama.cpp/convert_hf_to_gguf.py {merged_path} \
        --outfile gemma4-e4b-legal.f16.gguf \
        --outtype f16

    # Quantize to Q4_K_M if F16 conversion succeeded
    if os.path.exists("gemma4-e4b-legal.f16.gguf"):
        size_gb = os.path.getsize("gemma4-e4b-legal.f16.gguf") / 1024**3
        print(f"\nF16 GGUF created: {size_gb:.1f} GB")
        print("Quantizing to Q4_K_M...")
        !cd llama.cpp && make -j 2>/dev/null; ./llama-quantize ../gemma4-e4b-legal.f16.gguf ../gemma4-e4b-legal.Q4_K_M.gguf Q4_K_M
        if os.path.exists("gemma4-e4b-legal.Q4_K_M.gguf"):
            q4_size = os.path.getsize("gemma4-e4b-legal.Q4_K_M.gguf") / 1024**3
            print(f"Done! GGUF: gemma4-e4b-legal.Q4_K_M.gguf ({q4_size:.1f} GB)")
        else:
            print("Quantization failed. Use the Unsloth GGUF from the previous cell.")
    else:
        print("F16 conversion failed — Gemma 4 architecture may not be supported in llama.cpp yet.")
        print("Use the Unsloth GGUF export from the previous cell instead.")
        print("Unsloth has its own internal GGUF converter that handles Gemma 4.")

In [ ]:
import os

# === Create Adapter-Only Modelfile for Ollama ===
# WARNING: Ollama's ADAPTER directive only supports Gemma 1/2 SafeTensors.
# Gemma 3/4 adapters are NOT supported — use merged GGUF instead (cell above).
# This cell is kept as a reference / for future Ollama updates.

modelfile_adapter = """# WARNING: Ollama ADAPTER may not support Gemma 4 yet.
# Prefer: ollama create gemma4-legal -f Modelfile (with merged GGUF)
FROM gemma4:e4b
ADAPTER ./gemma4-e4b-legal-grpo-lora

PARAMETER temperature 0.7
PARAMETER num_ctx 8192

SYSTEM \"\"\"You are a specialized Legal AI Assistant powered by Gemma 4, fine-tuned with GRPO reinforcement learning for legal analysis. Citing specific statutes and case law is your priority.\"\"\"
"""

os.makedirs("gemma4-e4b-legal-grpo-lora", exist_ok=True)
with open("gemma4-e4b-legal-grpo-lora/Modelfile.adapter", "w") as f:
    f.write(modelfile_adapter)

# Zip the adapter directory
!zip -r gemma4-legal-final-adapter.zip gemma4-e4b-legal-grpo-lora/

print("\nAdapter package ready: gemma4-legal-final-adapter.zip")
print()
print("=== RECOMMENDED DEPLOYMENT ===")
print("  Use merged GGUF (from previous cells):")
print("    ollama create gemma4-legal:latest -f gemma4-e4b-legal/Modelfile")
print()
print("=== ALTERNATIVE (if Ollama adds Gemma 4 adapter support) ===")
print("  ollama create gemma4-legal:latest -f gemma4-e4b-legal-grpo-lora/Modelfile.adapter")
print()
print("=== OTHER OPTIONS ===")
print("  TRT-LLM:   trtllm-build --lora_dir gemma4-e4b-legal-grpo-lora/")
print("  HF Upload:  huggingface-cli upload <repo> gemma4-e4b-legal-grpo-lora/")

# Download
from google.colab import files
try:
    files.download('gemma4-legal-final-adapter.zip')
except Exception:
    print("Download from Colab file browser instead.")

## 14. Create Ollama Modelfile + TRT-LLM Template

In [ ]:
import glob
import os

# === GGUF Modelfile (if GGUF export succeeded) ===
gguf_files = glob.glob("gemma4-e4b-legal/*.gguf") + glob.glob("gemma4-e4b-legal/**/*.gguf")
gguf_path = gguf_files[0] if gguf_files else "gemma4-e4b-legal/unsloth.Q4_K_M.gguf"

modelfile_content = f"""FROM {gguf_path}

PARAMETER temperature 0.7
PARAMETER num_predict 2048
PARAMETER num_ctx 8192
PARAMETER top_k 40
PARAMETER top_p 0.9
PARAMETER repeat_penalty 1.1

SYSTEM \"\"\"You are a specialized Legal AI Assistant powered by Gemma 4, fine-tuned with GRPO reinforcement learning for legal analysis. You excel at:

- Legal document analysis with proper citation (Bluebook format)
- Evidence classification and chain of custody assessment
- Statute interpretation (U.S.C., CFR, state codes)
- Case law reasoning and precedent analysis
- Contract review and liability assessment

Always provide structured, well-reasoned analysis. Cite specific statutes and case law when relevant.\"\"\"

TEMPLATE \"\"\"{{{{{{ if .System }}}}}}<start_of_turn>system
{{{{{{ .System }}}}}}<end_of_turn>
{{{{{{ end }}}}}}{{{{{{ if .Prompt }}}}}}<start_of_turn>user
{{{{{{ .Prompt }}}}}}<end_of_turn>
<start_of_turn>model
{{{{{{ end }}}}}}{{{{{{ .Response }}}}}}<end_of_turn>\"\"\"
"""

os.makedirs("gemma4-e4b-legal", exist_ok=True)
with open("gemma4-e4b-legal/Modelfile", "w") as f:
    f.write(modelfile_content)

print("Modelfile written: gemma4-e4b-legal/Modelfile")
print()
print("Deploy on local machine:")
print("  GGUF:    ollama create gemma4-legal:latest -f gemma4-e4b-legal/Modelfile")
print("  Adapter: ollama create gemma4-legal:latest -f gemma4-e4b-legal-grpo-lora/Modelfile")
print()

# === TRT-LLM Build Template ===
trt_cmd = """
# TRT-LLM engine build command (run on your local GPU machine)
# Replace /path/to/base_model with local Gemma 4 E4B weights

trtllm-build --model_config /path/to/base_model/config.json \n
    --checkpoint_dir /path/to/base_model \n
    --output_dir ./gemma4-legal-engine \n
    --lora_dir ./gemma4-e4b-legal-grpo-lora \n
    --max_batch_size 4 \n
    --max_input_len 4096 \n
    --max_output_len 2048 \n
    --gemm_plugin auto \n
    --precision bfloat16
"""
print("TRT-LLM Build Template:")
print(trt_cmd)


## 15. Package for Download

In [ ]:
# Zip GGUF model for download
!zip -r gemma4-e4b-legal-gguf.zip gemma4-e4b-legal/

print("\nPackaged: gemma4-e4b-legal-gguf.zip (~3 GB)")
print("\nDownload and deploy:")
print("  1. Download gemma4-e4b-legal-gguf.zip")
print("  2. unzip gemma4-e4b-legal-gguf.zip")
print("  3. cd gemma4-e4b-legal/")
print("  4. ollama create gemma4-legal:latest -f Modelfile")
print("  5. Update .env: LLM_MODEL=gemma4-legal:latest")
print()
print("RTX 3060 Ti VRAM: ~3.5GB (Q4_K_M) + ~0.6GB (embeddinggemma) = ~4.1GB")
print("Remaining: ~4GB free for TensorRT / custom CUDA ops")

# Optional: Auto-download in Colab
# from google.colab import files
# files.download('gemma4-e4b-legal-gguf.zip')

## 16. (Optional) SFT Warm-up Before GRPO

If GRPO alone isn't producing strong enough results, you can do a
short SFT (Supervised Fine-Tuning) warm-up first, THEN apply GRPO.
This is the "SFT â†’ GRPO" two-stage pipeline used by DeepSeek-R1.

Run this cell BEFORE cells 8-10 if you want the two-stage approach.

In [ ]:
# OPTIONAL: SFT warm-up (run BEFORE GRPO training)
# Uncomment to enable

# from trl import SFTTrainer
# from transformers import TrainingArguments
#
# # Prepare SFT dataset (prompt-response pairs)
# def format_sft(example):
#     text = example.get('text', '')
#     instruction = "Analyze the following legal concept:"
#     return {
#         "conversations": [
#             {"role": "user", "content": instruction},
#             {"role": "assistant", "content": text}
#         ]
#     }
#
# sft_dataset = legal_dataset.map(format_sft, remove_columns=['text'], num_proc=4)
#
# sft_args = TrainingArguments(
#     output_dir="./gemma4-sft-warmup",
#     num_train_epochs=1,  # Just 1 epoch for warm-up
#     per_device_train_batch_size=2,
#     gradient_accumulation_steps=8,
#     learning_rate=2e-4,
#     fp16=not is_bfloat16_supported(),
#     bf16=is_bfloat16_supported(),
#     logging_steps=10,
#     save_strategy="no",  # Don't save SFT checkpoints
#     optim="adamw_8bit",
#     warmup_steps=50,
#     report_to="none",
# )
#
# sft_trainer = SFTTrainer(
#     model=model,
#     tokenizer=tokenizer,
#     train_dataset=sft_dataset,
#     max_seq_length=MAX_SEQ_LENGTH,
#     args=sft_args,
#     dataset_text_field="conversations",
#     packing=False,
# )
#
# print("SFT Warm-up (1 epoch)...")
# sft_trainer.train()
# print("SFT warm-up complete. Now run GRPO cells (8-10).")

---

## Summary

**What we trained**:
- Base: Gemma 4 E4B (4B active params, multimodal: text + vision + audio)
- Method: GRPO reinforcement learning with 7 legal reward functions
- Data: ~10K legal documents + codebase patterns + freeform reasoning + tool calling prompts
- Output: GGUF Q4_K_M (~3GB) for Ollama deployment

**GPU Tiers (auto-scaled)**:
| GPU | VRAM | BF16 TFLOPs | Batch | Generations | Est. Time |
|-----|------|-------------|-------|-------------|-----------|
| **G4 (Blackwell)** | 96 GB | 960 | 4 | 6 | **~1.5-3 hrs** |
| A100-80GB | 80 GB | 624 | 2 | 4 | ~3-4 hrs |
| A100-40GB | 40 GB | 312 | 1 | 3 | ~4-5 hrs |
| T4 | 15 GB | 65 | 1 | 2 | ~5-7 hrs |

**No PTX compatibility issues** â€” Unsloth uses Triton kernels that JIT-compile at runtime.
Any CUDA GPU (sm_70+) works without precompiled binaries.

**Reward Functions (7)**:
| Function | Signal |
|----------|--------|
| Citation accuracy | Bluebook, U.S.C., CFR formats |
| Reasoning chain | Logical connectors (therefore, because, thus) |
| Legal formatting | Numbered lists, section headers, legal terms |
| Web search grounding | Source attribution, URL refs, temporal hedging |
| Tool calling format | JSON tool call structure (name, arguments, query) |
| Anti-hallucination | Penalize fabricated case names |
| Length quality | Sweet spot: 50-300 words |

**G4 Blackwell Advantages**:
- 54% more BF16 compute than A100-80G â†’ faster forward/backward passes
- 20% more VRAM â†’ larger batch sizes (4 vs 2) and more generations (6 vs 4)
- More generations per prompt = better reward signal = higher quality LoRA
- Blackwell architecture (sm_100+) supports latest Triton/CUDA optimizations

**Deployment**:
1. Download `gemma4-e4b-legal-gguf.zip` (~3 GB)
2. `ollama create gemma4-legal:latest -f Modelfile`
3. Update `.env`: `LLM_MODEL=gemma4-legal:latest`
4. Update `ollama.ts`: `VLM_MODELS.legal = 'gemma4-legal:latest'`

**VRAM Budget (RTX 3060 Ti 8GB)**:
| Model | VRAM | Purpose |
|-------|------|---------|
| gemma4-legal (Q4_K_M) | ~3.5GB | Text + Vision LLM |
| embeddinggemma | ~0.6GB | 768-dim embeddings |
| CUDA ops (LibTorch) | ~0.5GB | Custom matrix computations |
| KV cache + overhead | ~1.5GB | Inference working memory |
| **Total** | **~6.1GB** | Fits in 8GB |

**Key Advantage**: Gemma 4 E4B does text + vision in ONE model.
Replaces the current dual-model setup (gemma3-legal for text + separate VLM pipeline).
The `vlm-evidence-analyzer.ts` can use the same `gemma4-legal:latest` model for both
text analysis and image analysis via Ollama's `images: [base64]` parameter.

**Existing merge pipeline works**:
```bash
./scripts/unsloth-training/merge-and-export.sh \
  --adapter gemma4-e4b-legal-grpo-lora \
  --base google/gemma-4-e4b-it \
  --name gemma4-legal
```

**Sources**:
- [Gemma 4 Model Card](https://ai.google.dev/gemma/docs/gemma4)
- [Unsloth GRPO Guide](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide)
- [GRPO Paper (DeepSeek)](https://arxiv.org/abs/2402.03300)
- [Unsloth GGUF Export](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)
- [Unsloth Ollama Export](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-ollama)
- [Colab G4 GPU Announcement](https://blog.google/technology/developers/colab-g4-gpu/) (RTX Pro 6000 Blackwell)

## 17. Merge LoRA into Base Model (Text-Only Adapter Surgery)

**Problem**: The original adapter had 884 tensors (588 language + 224 vision + 72 audio).
Despite `finetune_vision_layers=False`, generic `target_modules` (`q_proj`, etc.) matched
projections in ALL sub-models. The vision/audio tensors use `Gemma4ClippableLinear` which
PEFT doesn't recognize, causing all merge attempts to fail.

**Solution**: Adapter surgery — strip vision/audio tensors, keep only 588 language tensors.
Then use latest Unsloth (PR [#4807](https://github.com/unslothai/unsloth/pull/4807)) which
patches `PeftModel.from_pretrained` to handle `Gemma4ClippableLinear` → inner `.linear`.

**Two options**:
- **Option A**: Upload the surgically-cleaned adapter from local machine
- **Option B**: Pull from HF Hub and strip vision/audio tensors in-notebook

In [ ]:
import os
import json
from safetensors.torch import load_file, save_file

# === Option A: Upload text-only adapter from local ===
# Upload the gemma4-legal-text-only-adapter/ directory files
# (adapter_config.json + adapter_model.safetensors + tokenizer files)

# === Option B: Pull from HF Hub and strip in-notebook ===
ADAPTER_DIR = "gemma4-e4b-legal-grpo-lora"
TEXT_ONLY_DIR = "gemma4-e4b-legal-text-only-adapter"

# Check if we have the adapter from training (cells 28-29)
if os.path.exists(f"{ADAPTER_DIR}/adapter_model.safetensors"):
    print(f"Found adapter at {ADAPTER_DIR}/")
else:
    # Pull from HF Hub
    print("Downloading adapter from HF Hub...")
    from huggingface_hub import snapshot_download
    snapshot_download("Semaj90/gemma4-e4b-legal-grpo", local_dir=ADAPTER_DIR)
    print(f"Downloaded to {ADAPTER_DIR}/")

# Load and inspect
tensors = load_file(f"{ADAPTER_DIR}/adapter_model.safetensors")
keys = sorted(tensors.keys())
lang = {k: v for k, v in tensors.items() if "language_model" in k}
vis = [k for k in keys if "vision_tower" in k]
aud = [k for k in keys if "audio_tower" in k]

print(f"\nOriginal adapter: {len(keys)} tensors")
print(f"  language_model: {len(lang)}")
print(f"  vision_tower:   {len(vis)}")
print(f"  audio_tower:    {len(aud)}")

# Strip vision/audio if present
if vis or aud:
    os.makedirs(TEXT_ONLY_DIR, exist_ok=True)

    # Save language-only tensors
    save_file(lang, f"{TEXT_ONLY_DIR}/adapter_model.safetensors")

    # Copy config with exclude_modules added
    with open(f"{ADAPTER_DIR}/adapter_config.json") as f:
        config = json.load(f)
    config["exclude_modules"] = ["vision_tower.*", "audio_tower.*", "multi_modal_projector.*"]
    with open(f"{TEXT_ONLY_DIR}/adapter_config.json", "w") as f:
        json.dump(config, f, indent=2)

    # Copy tokenizer files
    import shutil
    for fname in os.listdir(ADAPTER_DIR):
        if fname not in ("adapter_model.safetensors", "adapter_config.json"):
            src = os.path.join(ADAPTER_DIR, fname)
            if os.path.isfile(src):
                shutil.copy2(src, os.path.join(TEXT_ONLY_DIR, fname))

    orig_mb = sum(v.nelement() * v.element_size() for v in tensors.values()) / 1024**2
    new_mb = sum(v.nelement() * v.element_size() for v in lang.values()) / 1024**2
    print(f"\nSurgery complete: {TEXT_ONLY_DIR}/")
    print(f"  {len(keys)} -> {len(lang)} tensors")
    print(f"  {orig_mb:.1f} MB -> {new_mb:.1f} MB (saved {orig_mb - new_mb:.1f} MB)")
    MERGE_ADAPTER_DIR = TEXT_ONLY_DIR
else:
    print("\nAdapter is already text-only — no surgery needed")
    MERGE_ADAPTER_DIR = ADAPTER_DIR

# Verify
t_check = load_file(f"{MERGE_ADAPTER_DIR}/adapter_model.safetensors")
bad = [k for k in t_check if "language_model" not in k]
assert len(bad) == 0, f"Found {len(bad)} non-language keys: {bad[:3]}"
print(f"\nVerified: {len(t_check)} language-only tensors in {MERGE_ADAPTER_DIR}/")
del tensors, lang, t_check

### 17b. Ensure Latest Unsloth (PR #4807 ClippableLinear Fix)

Unsloth merged [PR #4807](https://github.com/unslothai/unsloth/pull/4807) on April 3, 2026.
This patches `PeftModel.from_pretrained` to redirect `Gemma4ClippableLinear` to its inner
`.linear` attribute. Without this fix, PEFT refuses to load the adapter.

If you already installed Unsloth from git in cell 2, you have the fix. Otherwise:

In [ ]:
# Ensure latest Unsloth with Gemma 4 ClippableLinear fix (PR #4807)
import importlib
import unsloth
unsloth_version = getattr(unsloth, '__version__', 'unknown')
print(f"Current Unsloth: {unsloth_version}")

# If installed from git (cell 2), the fix is included.
# If version is older than 2026.4.3, reinstall:
if unsloth_version < "2026.4.3" and unsloth_version != "unknown":
    print("Updating Unsloth to get PR #4807 fix...")
    import subprocess
    subprocess.check_call([
        "pip", "install", "--upgrade", "--no-cache-dir",
        "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
    ])
    print("Restart the runtime after upgrade: Runtime -> Restart runtime")
    print("Then re-run from this cell (skip training cells 2-29)")
else:
    print("Unsloth version OK — PR #4807 fix should be included")

### 17c. Load Base Model + Merge Text-Only Adapter

Load the base model via Unsloth's `FastVisionModel`, then apply the text-only adapter.
Unsloth's loader handles the `Gemma4ClippableLinear` patching internally (PR #4807).

In [ ]:
import torch
import os

from unsloth import FastVisionModel, is_bfloat16_supported

MODEL_NAME = "unsloth/gemma-4-E4B-it-unsloth-bnb-4bit"
MAX_SEQ_LENGTH = 4096

print(f"Loading base model: {MODEL_NAME}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB
")

# Load base model in 4-bit
model, tokenizer = FastVisionModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

# === FIX: Replace Gemma4ClippableLinear with inner nn.Linear ===
# PEFT only supports nn.Linear, nn.Embedding, nn.Conv*d.
# Gemma4ClippableLinear wraps nn.Linear but PEFT doesn't recognize it.
# Since we're NOT putting LoRA on vision/audio towers, we can safely
# unwrap ClippableLinear -> inner .linear for those modules.
# This lets PEFT scan the model without crashing.

from transformers.models.gemma4.modeling_gemma4 import Gemma4ClippableLinear

replaced = 0
for name, module in model.named_modules():
    if isinstance(module, Gemma4ClippableLinear):
        # Navigate to parent and replace
        parts = name.split('.')
        parent = model
        for p in parts[:-1]:
            parent = getattr(parent, p)
        setattr(parent, parts[-1], module.linear)
        replaced += 1

print(f"Replaced {replaced} Gemma4ClippableLinear -> nn.Linear")

# Now PEFT can load the adapter without hitting ClippableLinear
print(f"
Applying text-only adapter from: {MERGE_ADAPTER_DIR}")
from peft import PeftModel
model = PeftModel.from_pretrained(model, MERGE_ADAPTER_DIR)

# Verify adapter loaded
lora_params = [n for n, p in model.named_parameters() if "lora_" in n]
print(f"LoRA parameters loaded: {len(lora_params)}")
print(f"  language_model: {sum(1 for n in lora_params if 'language_model' in n)}")
print(f"  vision_tower:   {sum(1 for n in lora_params if 'vision_tower' in n)}")
print(f"  audio_tower:    {sum(1 for n in lora_params if 'audio_tower' in n)}")

### 17d. Quick Inference Test (Pre-Merge Sanity Check)

Verify the adapter produces good legal analysis before committing to the merge.

In [ ]:
from transformers import TextStreamer

FastVisionModel.for_inference(model)
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

test_prompt = "Analyze 42 U.S.C. Section 1983 and explain when qualified immunity applies."
messages = [{"role": "user", "content": [{"type": "text", "text": test_prompt}]}]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

print(f"Prompt: {test_prompt}\n")
print("=" * 70)
model.generate(
    input_ids=inputs,
    streamer=text_streamer,
    max_new_tokens=512,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.1,
)
print("\n" + "=" * 70)
print("If the output shows legal citations and structured analysis, the adapter is working.")

### 17e. Dequantize + Merge LoRA + Export GGUF

**Key insight:** `merge_and_unload()` fails on 4-bit weights because PEFT tries to
add a 2D LoRA delta to flat NF4-compressed tensors (shape mismatch). The fix:
**dequantize all Linear4bit to BF16 BEFORE applying the adapter**, so PEFT sees
regular `nn.Linear` modules and merges cleanly.

Pipeline:
1. Load base model (4-bit on GPU)
2. Fix ClippableLinear wrappers
3. **Dequantize** every `Linear4bit` -> `nn.Linear` (BF16) **BEFORE adapter**
4. Apply LoRA adapter (PEFT sees regular Linear, creates vanilla LoRA layers)
5. `merge_and_unload()` (works because weights are now BF16)
6. Save clean safetensors + convert to GGUF

**Critical:** Do NOT install llama.cpp requirements.txt (downgrades torch to CPU-only).

In [ ]:
# === DEQUANTIZE -> APPLY ADAPTER -> MERGE -> SAVE ===
# Key: dequantize BEFORE adapter so merge_and_unload works on BF16 weights
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'safetensors', 'gguf', '--upgrade'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import torch, os, gc, json
import bitsandbytes as bnb
from safetensors.torch import save_file

CLEAN_DIR = 'gemma4-merged-clean'
os.makedirs(CLEAN_DIR, exist_ok=True)

# --- Step 1: Load base model (4-bit on GPU) ---
from unsloth import FastVisionModel
model, tokenizer = FastVisionModel.from_pretrained(
    'unsloth/gemma-4-E4B-it-unsloth-bnb-4bit',
    max_seq_length=4096,
    load_in_4bit=True,
)
print('Step 1/6: Base model loaded on GPU')

# --- Step 2: Fix ClippableLinear ---
from transformers.models.gemma4.modeling_gemma4 import Gemma4ClippableLinear
clip_count = 0
for name, module in list(model.named_modules()):
    if isinstance(module, Gemma4ClippableLinear):
        parts = name.split('.')
        parent = model
        for p in parts[:-1]:
            parent = getattr(parent, p)
        setattr(parent, parts[-1], module.linear)
        clip_count += 1
print(f'Step 2/6: Unwrapped {clip_count} ClippableLinear modules')

# --- Step 3: DEQUANTIZE all Linear4bit -> nn.Linear (BF16) BEFORE adapter ---
# This is the key fix: merge_and_unload() can't add LoRA deltas to 4-bit
# compressed weights (shape mismatch). Dequantize first so PEFT sees regular
# nn.Linear and the merge works cleanly.
deq_count = 0
for name, module in list(model.named_modules()):
    if isinstance(module, bnb.nn.Linear4bit):
        # Dequantize NF4 -> BF16
        w_deq = bnb.functional.dequantize_4bit(
            module.weight.data, module.weight.quant_state
        ).to(torch.bfloat16)

        # Replace with standard nn.Linear (keep on same device)
        new_linear = torch.nn.Linear(
            module.in_features, module.out_features,
            bias=(module.bias is not None),
            device=w_deq.device, dtype=torch.bfloat16
        )
        new_linear.weight = torch.nn.Parameter(w_deq)
        if module.bias is not None:
            new_linear.bias = torch.nn.Parameter(
                module.bias.data.to(torch.bfloat16)
            )

        # Splice into model
        parts = name.split('.')
        parent = model
        for p in parts[:-1]:
            parent = getattr(parent, p)
        setattr(parent, parts[-1], new_linear)
        deq_count += 1

print(f'Step 3/6: Dequantized {deq_count} layers from NF4 -> BF16')
print(f'  GPU memory: {torch.cuda.memory_allocated()/1024**3:.1f} GB allocated')

# --- Step 4: Apply LoRA adapter (now sees regular nn.Linear) ---
from peft import PeftModel
model = PeftModel.from_pretrained(model, 'Semaj90/gemma4-e4b-legal-grpo')
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Step 4/6: Adapter loaded ({trainable:,} trainable params)')

# --- Step 5: Merge LoRA into base weights (now BF16, merge works!) ---
model = model.merge_and_unload()
total = sum(p.numel() for p in model.parameters())
print(f'Step 5/6: LoRA merged! {total:,} total params')

# --- Step 6: Save clean safetensors ---
model = model.cpu()
torch.cuda.empty_cache()

sd = {}
for k, v in model.state_dict().items():
    # Strip any PEFT .base_layer. artifacts
    clean_k = k.replace('.base_layer.', '.')
    sd[clean_k] = v.to(torch.bfloat16) if v.is_floating_point() else v

save_file(sd, os.path.join(CLEAN_DIR, 'model.safetensors'))
tokenizer.save_pretrained(CLEAN_DIR)

# Config without quantization_config
from transformers import AutoConfig
cfg = AutoConfig.from_pretrained('google/gemma-4-E4B-it').to_dict()
cfg.pop('quantization_config', None)
cfg.pop('_name_or_path', None)
with open(os.path.join(CLEAN_DIR, 'config.json'), 'w') as f:
    json.dump(cfg, f, indent=2)

fsize = os.path.getsize(os.path.join(CLEAN_DIR, 'model.safetensors'))
print(f'Step 6/6: Saved {len(sd)} tensors ({fsize / 1024**3:.1f} GB) to {CLEAN_DIR}/')

# Verify: no 4-bit or PEFT artifacts
bad_shapes = {k: list(v.shape) for k, v in sd.items() if v.dim() == 2 and v.shape[1] == 1}
bad_keys = [k for k in sd if '.base_layer.' in k or '.absmax' in k or '.quant_state' in k]
if bad_shapes:
    print(f'WARNING: {len(bad_shapes)} tensors look like 4-bit (shape [N,1]): {list(bad_shapes.keys())[:3]}')
elif bad_keys:
    print(f'WARNING: {len(bad_keys)} PEFT/quant artifacts remain')
else:
    print('VERIFIED: Clean BF16 model - no 4-bit artifacts, no PEFT key artifacts')

del model; gc.collect(); torch.cuda.empty_cache()
print('\nReady for GGUF conversion (next cell)')

In [ ]:
# === GGUF CONVERSION (Q4_K_M for Ollama) ===
import os

CLEAN_DIR = 'gemma4-merged-clean'
GGUF_DIR = 'gemma4-gguf'
os.makedirs(GGUF_DIR, exist_ok=True)

# Clone llama.cpp (converter only, no build needed for convert step)
if not os.path.exists('llama.cpp'):
    !git clone --depth 1 https://github.com/ggerganov/llama.cpp.git

# CRITICAL: Only install gguf package, NOT requirements.txt
# requirements.txt downgrades torch to CPU-only and transformers to 4.x
%pip install -q gguf --upgrade

# Step 1: Convert safetensors -> BF16 GGUF
bf16_path = os.path.join(GGUF_DIR, 'gemma4-e4b-legal.bf16.gguf')
print('Converting merged BF16 safetensors -> BF16 GGUF...')
!python llama.cpp/convert_hf_to_gguf.py {CLEAN_DIR} --outfile {bf16_path} --outtype bf16

if not os.path.exists(bf16_path):
    raise FileNotFoundError(f'BF16 GGUF conversion failed - check output above')

bf16_gb = os.path.getsize(bf16_path) / 1024**3
print(f'\nBF16 GGUF: {bf16_gb:.1f} GB')

# Step 2: Build llama-quantize and quantize to Q4_K_M
print('\nBuilding llama-quantize...')
!cd llama.cpp && cmake -B build -DCMAKE_BUILD_TYPE=Release 2>&1 | tail -2
!cd llama.cpp && cmake --build build --target llama-quantize -j$(nproc) 2>&1 | tail -3

q4_path = os.path.join(GGUF_DIR, 'gemma4-e4b-legal.Q4_K_M.gguf')
print(f'\nQuantizing BF16 -> Q4_K_M...')
!./llama.cpp/build/bin/llama-quantize {bf16_path} {q4_path} Q4_K_M

if os.path.exists(q4_path):
    q4_gb = os.path.getsize(q4_path) / 1024**3
    print(f'\nQ4_K_M GGUF: {q4_gb:.1f} GB - ready for Ollama!')
    # Remove BF16 intermediate to save disk
    os.remove(bf16_path)
    print(f'Removed BF16 intermediate ({bf16_gb:.1f} GB freed)')
else:
    print('Quantization failed - BF16 GGUF still available for manual quantization')

In [ ]:
# === MODELFILE + PACKAGING FOR DOWNLOAD ===
import os, shutil

GGUF_DIR = 'gemma4-gguf'
CLEAN_DIR = 'gemma4-merged-clean'
FINAL_DIR = 'gemma4-e4b-legal-final'
os.makedirs(FINAL_DIR, exist_ok=True)

# Find the GGUF file
gguf_file = None
for f in os.listdir(GGUF_DIR):
    if f.endswith('.gguf'):
        gguf_file = os.path.join(GGUF_DIR, f)
        break

if gguf_file:
    gguf_name = os.path.basename(gguf_file)
    shutil.copy2(gguf_file, os.path.join(FINAL_DIR, gguf_name))
    print(f'Copied {gguf_name} to {FINAL_DIR}/')
else:
    print('WARNING: No GGUF file found - packaging safetensors only')
    gguf_name = None

# Write Ollama Modelfile
modelfile = f'''FROM ./{gguf_name if gguf_name else 'model.gguf'}

PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER top_k 40
PARAMETER repeat_penalty 1.1
PARAMETER num_ctx 8192
PARAMETER stop "<end_of_turn>"
PARAMETER stop "<eos>"

TEMPLATE """<start_of_turn>system
You are a legal AI assistant specialized in U.S. law. Provide accurate legal analysis with proper Bluebook citations. Structure responses with clear headings, numbered elements, and relevant statutory/case references.
<end_of_turn>
<start_of_turn>user
{{ .Prompt }}
<end_of_turn>
<start_of_turn>model
{{ .Response }}
<end_of_turn>"""

SYSTEM """You are a legal AI assistant fine-tuned on U.S. federal and state law. Always cite statutes (U.S.C., CFR) and case law in Bluebook format. Identify relevant legal elements, potential defenses, and procedural considerations."""
'''

with open(os.path.join(FINAL_DIR, 'Modelfile'), 'w') as f:
    f.write(modelfile)
print(f'Modelfile written to {FINAL_DIR}/Modelfile')

# Package for download
print('\n--- Packaging for download ---')

# 1. GGUF package (for Ollama)
if gguf_name:
    !cd {FINAL_DIR} && zip -0 ../gemma4-e4b-legal-gguf.zip {gguf_name} Modelfile
    gguf_zip = 'gemma4-e4b-legal-gguf.zip'
    if os.path.exists(gguf_zip):
        print(f'GGUF zip: {os.path.getsize(gguf_zip)/1024**3:.1f} GB')

# 2. Safetensors package (for HF Hub / inference pipeline)
!cd {CLEAN_DIR} && zip -0 ../gemma4-e4b-legal-safetensors.zip model.safetensors config.json tokenizer* special_tokens*
st_zip = 'gemma4-e4b-legal-safetensors.zip'
if os.path.exists(st_zip):
    print(f'Safetensors zip: {os.path.getsize(st_zip)/1024**3:.1f} GB')

print('\n--- Download instructions ---')
print('from google.colab import files')
if gguf_name:
    print(f'files.download("gemma4-e4b-legal-gguf.zip")       # GGUF for Ollama')
print(f'files.download("gemma4-e4b-legal-safetensors.zip")  # Safetensors for HF Hub')
print('\n--- Ollama usage (after download) ---')
print('  1. unzip gemma4-e4b-legal-gguf.zip')
print('  2. ollama create gemma4-legal:latest -f Modelfile')
print('  3. ollama run gemma4-legal:latest')

### 17f. Upload Merged Model + Text-Only Adapter to HF Hub

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

# Upload text-only adapter (replaces the one with vision/audio contamination)
print("Uploading text-only adapter to Semaj90/gemma4-e4b-legal-grpo...")
api.upload_folder(
    folder_path=MERGE_ADAPTER_DIR,
    repo_id="Semaj90/gemma4-e4b-legal-grpo",
    commit_message="Replace with text-only adapter (stripped vision/audio tower tensors)",
)
print("Text-only adapter uploaded!")

# Upload GGUF to a separate repo
GGUF_REPO = "Semaj90/gemma4-e4b-legal-grpo-GGUF"
print(f"\nUploading GGUF to {GGUF_REPO}...")
try:
    api.create_repo(GGUF_REPO, exist_ok=True)
    api.upload_folder(
        folder_path=GGUF_DIR,
        repo_id=GGUF_REPO,
        commit_message="Gemma 4 E4B Legal GRPO — Q4_K_M GGUF for Ollama",
    )
    print(f"GGUF uploaded to {GGUF_REPO}")
except Exception as e:
    print(f"GGUF upload failed: {e}")
    print("Upload manually: huggingface-cli upload Semaj90/gemma4-e4b-legal-grpo-GGUF " + GGUF_DIR)

print("\n=== HF Hub Repos ===")
print(f"  Adapter: https://huggingface.co/Semaj90/gemma4-e4b-legal-grpo")
print(f"  GGUF:    https://huggingface.co/{GGUF_REPO}")